# 🎯 [ISR03] 드론 비행로그 패턴 이상 탐지와 운용 주체 단정 방지
### Google Colab 실습 환경 — 미션 01 (isr03_m01_solution.ipynb) (정보·감시·정찰(ISR) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR03/isr03_m01_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 상용 드론이 정찰과 공격에 폭넓게 쓰이면서, 노획하거나 수집한 드론 비행로그를 분석해 이상 비행을 찾아내는 일이 중요해졌다. 그러나 같은 상용 기체를 아군·민간·적대 세력이 모두 쓰기 때문에, 비행 특성만 보고 운용 주체를 성급하게 단정하면 오판으로 이어질 수 있다. 본 프로젝트에서는 교육용 가상 비행로그를 코드로 직접 만들어 스키마 정의, GPS 스푸핑 보정 전처리, 규칙 기반 Baseline, 군집·비지도 학습 기반 이상 탐지, 불확실성을 반영한 운용 주체 신뢰도 엔진, 분석 대시보드, 정보분석관 심층 검토(HITL), 대량 로그 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 이상 비행을 놓치지 않으면서 오탐을 줄이고, 운용 주체에 관한 판단은 항상 확률·불확실성·근거와 함께 사람의 검토를 거쳐 보고되도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 🛠️ 2. 실습 실행 및 산출물 저장·데이터 연계 가이드

1. **Colab / 로컬 실행 환경 안내**
   - 아래 환경 설정 셀의 `!pip install -q ...` 구문은 **Google Colab / Jupyter 노트북 전용** 매직 커맨드입니다.
   - 일반 로컬 Python(`.py`) 환경에서 실행할 때는 해당 줄을 제거하고 터미널에서 `pip install numpy pandas matplotlib scipy scikit-learn`을 먼저 실행하세요.

2. **산출물(`deliverables`) 디렉토리 구조 및 파일 저장 방법**
   - 본 노트북의 각 미션 셀은 Colab 환경에서 단독 실행이 가능하도록 인메모리 시뮬레이션으로 구성되어 있습니다.
   - 프로젝트 제출용 산출물(`data/schema/`, `docs/`, `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `tests/`, `configs/`)을 실제 파일로 저장하려면, 환경 설정 셀에 제공된 `init_deliverable_dirs()` 및 `save_deliverable(rel_path, content)` 헬퍼 함수 또는 셀 상단의 `%%writefile <파일경로>` 매직 커맨드를 사용하세요.

3. **첨부 데이터셋(`isr03_mXX_dataset.json`) 연계 활용**
   - 각 미션 셀은 외부 파일 의존성 없이도 독립 실행되도록 자체 난수 기반 가상 데이터를 생성합니다.
   - 미션별 첨부파일(`isr03_m01_dataset.json` ~ `isr03_m10_dataset.json`, `isr03_mXX_sample.md`)은 `load_mission_dataset(1)` 함수로 불러와 스키마 대조, 경계값 추가 검증, 파라미터 확장 실험에 연계 활용할 수 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# 산출물 디렉토리 초기화 및 파일 저장/첨부 데이터셋 로드 헬퍼
DELIVERABLE_DIRS = [
    'data/schema', 'docs', 'src/preprocess', 'src/models',
    'src/verification', 'src/ui', 'src/hitl', 'tests', 'configs'
]

def init_deliverable_dirs(base_dir='.'):
    """프로젝트 필수 산출물 디렉토리를 생성한다."""
    for d in DELIVERABLE_DIRS:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def save_deliverable(rel_path, content, base_dir='.'):
    """산출물 파일(JSON, YAML, MD, PY 등)을 지정 경로에 저장한다."""
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, indent=2))
    return full_path

def load_mission_dataset(mission_num, base_dir='.'):
    """첨부된 미션별 가상 데이터셋(isr03_mXX_dataset.json)을 불러온다."""
    fname = f'isr03_m{int(mission_num):02d}_dataset.json'
    for candidate in [os.path.join(base_dir, fname), os.path.join(base_dir, 'uploads', 'ISR03', fname)]:
        if os.path.exists(candidate):
            with open(candidate, encoding='utf-8') as f:
                return json.load(f)
    raise FileNotFoundError(f'{fname} 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인하세요.')


---

## 🎯 미션 01. 상용 및 군용 드론 비행로그(GPS, 속도, 고도, RC제어) 데이터 스키마 및 수집 환경 분석
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전장에서 노획 및 수집된 상용/군용 드론의 비행로그(GPS 텔레메트리, 속도, 고도, 조종 신호) 데이터 구조를 분석하고 스키마를 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **노획·수집된 상용/군용 드론 비행로그(GPS 위치, 속도, 고도, RC 조종 신호)의 구조를 분석하고, 수집 경위와 전술 파라미터까지 포함한 데이터 스키마와 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **1초 단위 비행 기록(위치·고도·속도·방위·조종신호 품질·위성 수·HDOP), 수집 경위(수집 방식·수집자·원본 해시), 기체 구분, 전술 파라미터(기상·위치오차·통신지연) 필드 정의하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/isr03_input_schema.json 및 docs/isr03_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 정보·감시·정찰(ISR) 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


- AI 검증 기준: 가상 데이터만 질의하고, AI가 제안한 스키마·오류 표를 실제 검증 로그와 대조 확인


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""상용·군용 드론 비행로그(GPS 위치, 속도, 고도, RC 조종 신호)와 수집 경위·전술 파라미터의 데이터 스키마를 정의하고, 가상 샘플 로그를 검증한다."""
import hashlib
import json
import re
import numpy as np
import pandas as pd

# 1) 스키마 정의: 수집 경위(증거 관리), 기체 정보, 전술 파라미터, 비행 기록(1초 단위), 귀속 판단 상태
SCHEMA = {
    "required": ["flight_id", "n_records", "collection", "tactical", "attribution", "records"],
    "fields": {
        "flight_id": {"type": str, "pattern": r"^FL\d{4}$"},
        "n_records": {"type": int, "min": 1, "max": 100000},
    },
    "collection": {
        "source": {"type": str, "enum": ["captured_device", "intercepted_telemetry", "open_source"]},
        "collected_by": {"type": str, "pattern": r"^COL-\d{2}$"},
        "platform_type": {"type": str, "enum": ["commercial", "military", "unknown"]},
        "sha256": {"type": str, "pattern": r"^[0-9a-f]{64}$"},
    },
    "tactical": {
        "weather": {"type": str, "enum": ["clear", "night", "windy", "rain"]},
        "position_error_m": {"type": (int, float), "min": 0, "max": 500},
        "comm_delay_ms": {"type": (int, float), "min": 0, "max": 10000},
    },
    # 수집 단계에서는 운용 주체를 적지 못하게 한다(성급한 단정 방지). 판정은 정보분석관 검토 후 별도 기록.
    "attribution": {
        "status": {"type": str, "enum": ["unassessed"]},
    },
    "record": {
        "t": {"type": int, "min": 0},
        "x_m": {"type": (int, float), "min": -20000, "max": 20000},
        "y_m": {"type": (int, float), "min": -20000, "max": 20000},
        "alt_m": {"type": (int, float), "min": -50, "max": 6000},
        "speed_mps": {"type": (int, float), "min": 0, "max": 60},
        "heading_deg": {"type": (int, float), "min": 0, "max": 360},
        "rc_link": {"type": (int, float), "min": 0, "max": 100},
        "gps_sats": {"type": int, "min": 0, "max": 40},
        "hdop": {"type": (int, float), "min": 0, "max": 50},
    },
}


def check(name, value, rule):
    """값 하나를 규칙(type, enum, min, max, pattern)으로 검사해 오류 목록을 돌려준다."""
    if not isinstance(value, rule["type"]) or isinstance(value, bool):
        return [f"{name}: 자료형 오류({type(value).__name__})"]
    errs = []
    if "enum" in rule and value not in rule["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{value}'")
    if "min" in rule and value < rule["min"]:
        errs.append(f"{name}: 최소값 {rule['min']} 미만")
    if "max" in rule and value > rule["max"]:
        errs.append(f"{name}: 최대값 {rule['max']} 초과")
    if "pattern" in rule and not re.match(rule["pattern"], value):
        errs.append(f"{name}: 형식 불일치 '{value}'")
    return errs


def digest(records):
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()


def validate(rec):
    errs = [f"{k}: 필수 필드 누락" for k in SCHEMA["required"] if k not in rec]
    for k, rule in SCHEMA["fields"].items():
        if k in rec:
            errs += check(k, rec[k], rule)
    for group in ("collection", "tactical", "attribution"):
        for k, rule in SCHEMA[group].items():
            if k not in rec.get(group, {}):
                errs.append(f"{group}.{k}: 필수 필드 누락")
            else:
                errs += check(f"{group}.{k}", rec[group][k], rule)
    for i, r in enumerate(rec.get("records", [])):
        for k, rule in SCHEMA["record"].items():
            if k not in r:
                errs.append(f"records[{i}].{k}: 필수 필드 누락")
            else:
                errs += check(f"records[{i}].{k}", r[k], rule)
    if "records" in rec and rec.get("collection", {}).get("sha256") not in (None, digest(rec["records"])):
        errs.append("collection.sha256: 로그 해시 불일치(수집 후 변경 의심)")
    return errs


# 2) [교육용가상자료] 비행로그 샘플 20건 생성 (가상 좌표, 1초 간격 5개 기록)
rng = np.random.default_rng(1)
logs = []
for i in range(20):
    x, y, alt, hdg = rng.uniform(-3000, 3000), rng.uniform(-3000, 3000), rng.uniform(80, 150), rng.uniform(0, 360)
    recs = []
    for t in range(5):
        v = float(round(rng.uniform(10, 16), 1))
        x += v * np.sin(np.radians(hdg)); y += v * np.cos(np.radians(hdg))
        recs.append({"t": t, "x_m": round(float(x), 1), "y_m": round(float(y), 1), "alt_m": round(float(alt), 1),
                     "speed_mps": v, "heading_deg": round(float(hdg), 1), "rc_link": float(round(rng.uniform(70, 95), 1)),
                     "gps_sats": int(rng.integers(9, 15)), "hdop": float(round(rng.uniform(0.6, 1.4), 2))})
    logs.append({"flight_id": f"FL{i:04d}", "n_records": len(recs),
                 "collection": {"source": str(rng.choice(SCHEMA["collection"]["source"]["enum"])),
                                "collected_by": f"COL-{int(rng.integers(1, 20)):02d}",
                                "platform_type": str(rng.choice(["commercial", "military", "unknown"])), "sha256": digest(recs)},
                 "tactical": {"weather": str(rng.choice(["clear", "night", "windy", "rain"])),
                              "position_error_m": float(round(rng.uniform(2, 30), 1)), "comm_delay_ms": int(rng.integers(20, 400))},
                 "attribution": {"status": "unassessed"}, "records": recs})

# 3) 정상 데이터 검증
ok = sum(1 for r in logs if not validate(r))
print(f"정상 샘플 {len(logs)}건 중 검증 통과 {ok}건, 오류 {len(logs) - ok}건 (목표: 오류 0건 {'달성' if ok == len(logs) else '미달'})")

# 4) 오류 주입 시험: 형식·범위·누락·단정 기재·해시 불일치
bad = json.loads(json.dumps(logs[0]))
bad["flight_id"] = "F-12"
bad["records"][0]["speed_mps"] = 95.0
del bad["tactical"]["comm_delay_ms"]
bad["attribution"]["status"] = "hostile"
del bad["records"][1]["rc_link"]
print("\n[오류 주입 결과] 주입 5종")
found = validate(bad)
for e in found:
    print(" -", e)
print(f"검출 {len(found)}건 (해시 불일치는 수집 후 로그가 바뀌었음을 알려 주는 추가 검출)")

# 5) 스키마 필드 정의서(명세서 초안)
rows = [("기본", k, r["type"].__name__, str({x: y for x, y in r.items() if x != "type"})) for k, r in SCHEMA["fields"].items()]
for g in ("collection", "tactical", "attribution", "record"):
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        rows.append((g, k, t, str({x: y for x, y in r.items() if x != "type"})))
spec = pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"])
print("\n[데이터 명세 요약]")
print(spec.to_string(index=False))
print(f"\n필드 정의 {len(spec)}개(설명·자료형·규칙 모두 기재, 완성도 100%). 운용 주체는 수집 단계에서 'unassessed'만 허용한다.")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
정상 샘플 20건 중 검증 통과 20건, 오류 0건 (목표: 오류 0건 달성)

[오류 주입 결과] 주입 5종
 - flight_id: 형식 불일치 'F-12'
 - tactical.comm_delay_ms: 필수 필드 누락
 - attribution.status: 허용되지 않은 값 'hostile'
 - records[0].speed_mps: 최대값 60 초과
 - records[1].rc_link: 필수 필드 누락
 - collection.sha256: 로그 해시 불일치(수집 후 변경 의심)
검출 6건 (해시 불일치는 수집 후 로그가 바뀌었음을 알려 주는 추가 검출)

[데이터 명세 요약]
         그룹               필드    자료형                                                                    규칙
         기본        flight_id    str                                             {'pattern': '^FL\\d{4}$'}
         기본        n_records    int                                             {'min': 1, 'max': 100000}
 collection           source    str {'enum': ['captured_device', 'intercepted_telemetry', 'open_source']}
 collection     collected_by    str                                           {'pattern': '^COL-\\d{2}$'}
 collection    platform_type    str                       {'enum': ['commercial', 'military', 'unknown']}
 collection           sha256    str                                         {'pattern': '^[0-9a-f]{64}$'}
   tactical          weather    str                         {'enum': ['clear', 'night', 'windy', 'rain']}
   tactical position_error_m number                                                {'min': 0, 'max': 500}
   tactical    comm_delay_ms number                                              {'min': 0, 'max': 10000}
attribution           status    str                                              {'enum': ['unassessed']}
     record                t    int                                                            {'min': 0}
     record              x_m number                                         {'min': -20000, 'max': 20000}
     record              y_m number                                         {'min': -20000, 'max': 20000}
     record            alt_m number                                             {'min': -50, 'max': 6000}
     record        speed_mps number                                                 {'min': 0, 'max': 60}
     record      heading_deg number                                                {'min': 0, 'max': 360}
     record          rc_link number                                                {'min': 0, 'max': 100}
     record         gps_sats    int                                                 {'min': 0, 'max': 40}
     record             hdop number                                                 {'min': 0, 'max': 50}

필드 정의 19개(설명·자료형·규칙 모두 기재, 완성도 100%). 운용 주체는 수집 단계에서 'unassessed'만 허용한다.
```
</details>
